In [1]:
from google.colab import files

uploaded = files.upload()

Saving luinab_provisional_scored_households_v2.csv to luinab_provisional_scored_households_v2.csv


In [2]:
!pip -q install pandas scikit-learn joblib

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from pathlib import Path

print("Libraries ready.")

Libraries ready.


In [3]:
CSV_PATH = "/content/luinab_provisional_scored_households_v2.csv"

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nTarget distribution:")
print(df["vulnerability_level"].value_counts(dropna=False))

print("\nReview status:")
print(df["label_status"].value_counts(dropna=False))

display(df.head())

Dataset shape: (995, 29)

Columns:
['hh_id', 'member_count', 'head_record_count', 'head_education', 'household_monthly_income', 'source_family_size', 'family_size', 'income_conflict', 'family_size_source_conflict', 'family_size_mismatch', 'all_members_explicitly_unemployed', 'provisional_disability_indicator', 'disability_review_matches', 'income_points', 'family_size_points', 'employment_points', 'education_points', 'provisional_disability_points', 'total_score', 'vulnerability_level', 'label_status', 'head_record_review_required', 'income_review_required', 'family_size_review_required', 'disability_review_required', 'needs_review', 'review_reasons', 'illness_indicator', 'illness_points']

Target distribution:
vulnerability_level
Medium    611
Low       293
High       91
Name: count, dtype: int64

Review status:
label_status
PROVISIONAL — RUBRIC-GENERATED    995
Name: count, dtype: int64


,hh_id,member_count,head_record_count,head_education,household_monthly_income,source_family_size,family_size,income_conflict,family_size_source_conflict,family_size_mismatch,...,vulnerability_level,label_status,head_record_review_required,income_review_required,family_size_review_required,disability_review_required,needs_review,review_reasons,illness_indicator,illness_points
0,1,5,1,CG,30000.0,5.0,5,False,False,False,...,Medium,PROVISIONAL — RUBRIC-GENERATED,False,False,False,False,False,NaN,True,3
1,2,4,1,CG,45000.0,4.0,4,False,False,False,...,Low,PROVISIONAL — RUBRIC-GENERATED,False,False,False,False,False,NaN,False,0
2,3,5,1,CG,30000.0,5.0,5,False,False,False,...,Low,PROVISIONAL — RUBRIC-GENERATED,False,False,False,False,False,NaN,False,0
3,4,4,1,CG,30000.0,4.0,4,False,False,False,...,Low,PROVISIONAL — RUBRIC-GENERATED,False,False,False,False,False,NaN,False,0
4,5,7,1,CG,158000.0,7.0,7,False,False,False,...,Medium,PROVISIONAL — RUBRIC-GENERATED,False,False,False,True,True,Review illness terms: MYASTHENIA GRAVIS,True,3


In [4]:
TARGET = "vulnerability_level"

# Inputs available for household profiling and prediction.
FEATURES = [
    "member_count",
    "head_education",
    "household_monthly_income",
    "family_size",
    "all_members_explicitly_unemployed",
    "provisional_disability_indicator",
    "illness_indicator",
]

required_columns = FEATURES + [TARGET]
missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Remove records without a target label.
df = df.dropna(subset=[TARGET]).copy()

# Standardize target text.
df[TARGET] = df[TARGET].astype(str).str.strip()

valid_labels = {"High", "Medium", "Low"}
unexpected = set(df[TARGET].unique()) - valid_labels

if unexpected:
    raise ValueError(f"Unexpected vulnerability labels: {unexpected}")

X = df[FEATURES].copy()
y = df[TARGET].copy()

# Convert Boolean-like values into consistent categories.
boolean_columns = [
    "all_members_explicitly_unemployed",
    "provisional_disability_indicator",
    "illness_indicator",
]

for col in boolean_columns:
    X[col] = (
        X[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({
            "true": "Yes",
            "1": "Yes",
            "yes": "Yes",
            "false": "No",
            "0": "No",
            "no": "No",
            "nan": pd.NA,
            "none": pd.NA,
            "": pd.NA,
        })
    )

numeric_features = [
    "member_count",
    "household_monthly_income",
    "family_size",
]

categorical_features = [
    "head_education",
    "all_members_explicitly_unemployed",
    "provisional_disability_indicator",
    "illness_indicator",
]

print("Rows:", len(X))
print("Features:", FEATURES)
print("\nLabels:")
print(y.value_counts())

display(X.head())

Rows: 995
Features: ['member_count', 'head_education', 'household_monthly_income', 'family_size', 'all_members_explicitly_unemployed', 'provisional_disability_indicator', 'illness_indicator']

Labels:
vulnerability_level
Medium    611
Low       293
High       91
Name: count, dtype: int64


,member_count,head_education,household_monthly_income,family_size,all_members_explicitly_unemployed,provisional_disability_indicator,illness_indicator
0,5,CG,30000.0,5,No,No,Yes
1,4,CG,45000.0,4,No,No,No
2,5,CG,30000.0,5,No,No,No
3,4,CG,30000.0,4,No,No,No
4,7,CG,158000.0,7,No,No,Yes


In [5]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

model_pipeline.fit(X_train, y_train)

print("\nTraining completed.")

Training rows: 796
Testing rows: 199

Training completed.


In [6]:
predictions = model_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"Holdout accuracy: {accuracy:.4f} ({accuracy:.1%})")

print("\nClassification report:")
print(classification_report(
    y_test,
    predictions,
    labels=["High", "Medium", "Low"],
    zero_division=0,
))

print("\nConfusion matrix:")
print(pd.DataFrame(
    confusion_matrix(
        y_test,
        predictions,
        labels=["High", "Medium", "Low"],
    ),
    index=["Actual High", "Actual Medium", "Actual Low"],
    columns=["Predicted High", "Predicted Medium", "Predicted Low"],
))

Holdout accuracy: 0.9849 (98.5%)

Classification report:
              precision    recall  f1-score   support

        High       0.90      1.00      0.95        18
      Medium       1.00      0.98      0.99       122
         Low       0.98      1.00      0.99        59

    accuracy                           0.98       199
   macro avg       0.96      0.99      0.98       199
weighted avg       0.99      0.98      0.99       199


Confusion matrix:
               Predicted High  Predicted Medium  Predicted Low
Actual High                18                 0              0
Actual Medium               2               119              1
Actual Low                  0                 0             59


In [7]:
class_counts = y.value_counts()
n_splits = min(5, int(class_counts.min()))

if n_splits >= 2:
    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42,
    )

    scores = cross_val_score(
        model_pipeline,
        X,
        y,
        cv=cv,
        scoring="f1_macro",
        n_jobs=1,
    )

    print("Cross-validation macro F1 scores:", scores.round(3))
    print("Mean macro F1:", round(scores.mean(), 3))
    print("Standard deviation:", round(scores.std(), 3))
else:
    print("Not enough records in every class for stratified cross-validation.")

Cross-validation macro F1 scores: [0.969 0.938 0.98  0.962 0.975]
Mean macro F1: 0.965
Standard deviation: 0.015


In [8]:
MODEL_PATH = "/content/luinab_household_rf_pipeline.joblib"

artifact = {
    "model": model_pipeline,
    "feature_names": FEATURES,
    "target_name": TARGET,
    "classes": list(model_pipeline.named_steps["classifier"].classes_),
    "version": "1.0",
    "training_rows": len(X_train),
    "test_accuracy": float(accuracy),
}

joblib.dump(artifact, MODEL_PATH)

print("Saved:", MODEL_PATH)

from google.colab import files
files.download(MODEL_PATH)

Saved: /content/luinab_household_rf_pipeline.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
sample_household = pd.DataFrame([{
    "member_count": 5,
    "head_education": "CG",
    "household_monthly_income": 8000,
    "family_size": 5,
    "all_members_explicitly_unemployed": "No",
    "provisional_disability_indicator": "No",
    "illness_indicator": "No",
}])

predicted_class = model_pipeline.predict(sample_household)[0]
probabilities = model_pipeline.predict_proba(sample_household)[0]
classes = model_pipeline.named_steps["classifier"].classes_

print("Predicted vulnerability:", predicted_class)
print("Class probabilities:")

for class_name, probability in zip(classes, probabilities):
    print(f"{class_name}: {probability:.1%}")

Predicted vulnerability: Medium
Class probabilities:
High: 9.8%
Low: 20.0%
Medium: 70.2%


In [10]:
import pandas as pd